# สำรวจข้อมูล 3/4: ค่าผิดปกติ และรูปแบบการขาดหาย (Outliers / Missing patterns)

| ไฟล์ | เกณฑ์ 3.5.2 (18 คะแนน) |
|---|---|
| ✅ eda_step1_descriptive | 3.5.2.1 สถิติเบื้องต้น (8) |
| ✅ eda_step2_trends | 3.5.2.2 แนวโน้ม/ความสัมพันธ์ (5) |
| **→** eda_step3_outliers_missing | 3.5.2.3 ค่าผิดปกติ/การขาดหาย (3) |
|  eda_step4_insights | 3.5.2.4 Insight + ตอบคำถามจาก step1_questions (2) |

ตอบเกณฑ์ 3.5.2.3 (3 คะแนน) ครบ 3 ข้อ:
1. ตรวจจับค่าผิดปกติด้วยเกณฑ์ชัดเจน (IQR / Z-score / เหตุผลธุรกิจ) และ **ชี้ว่าตัวแปรไหนได้รับผลกระทบ**
2. รูปแบบการขาดหาย: % ว่างต่อคอลัมน์ + ภาพที่แสดง **การขาดร่วมกัน**
3. วิธีรับมือและผลกระทบ

> ต่างจากขั้นทำความสะอาดที่ 3 อย่างไร: ขั้นนั้น **ตัดสินใจ** ว่าค่าไหนผิดจริง ส่วนไฟล์นี้ **สำรวจ** ว่าค่าผิดปกติที่เหลืออยู่ (ซึ่งเป็นค่าจริง) กระทบการวิเคราะห์แค่ไหน และตรวจครบทุกตัวแปรสำคัญ ไม่ใช่แค่ไฟฟ้าต่อคน

> รัน **Kernel → Restart Kernel and Run All Cells**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DATA = os.path.join("final", "energy_clean.csv")
if not os.path.exists(DATA):
    raise FileNotFoundError("ไม่พบ final/energy_clean.csv ให้รันขั้นทำความสะอาด 1-5 ให้สำเร็จก่อน")
OUT = "eda_output"                       # เก็บรูปและตารางไว้ใช้ทำสไลด์
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(DATA)
LATEST = int(df.loc[df["year_complete"], "year"].max())   # ปีล่าสุดที่ข้อมูลครบ = 2021
y = df[df["year"] == LATEST].copy()                        # ภาพตัดขวาง (ทุกประเทศ ปีเดียว)
ASEAN = sorted(df.loc[df["is_asean"], "country"].unique())
FUEL_COLOR = {"coal": "#4A4039", "oil": "#8C5A3C", "gas": "#E0913A", "nuclear": "#7B61A8", "hydro": "#2E78AD",
              "wind": "#5FB3A1", "solar": "#E8B923", "biofuel": "#7A9A3A", "other_renewables": "#9AA5A8"}

def savefig(name):
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")

print(f"ข้อมูล: {df.shape[0]:,} แถว x {df.shape[1]} คอลัมน์ | {df['country'].nunique()} ประเทศ | ปี {df['year'].min()}-{df['year'].max()}")
print(f"ปีที่ใช้วิเคราะห์ภาพตัดขวาง: {LATEST} ({len(y)} ประเทศ)")

## 3.1 ตรวจค่าผิดปกติทุกตัวแปรสำคัญ (ปี 2021)
ใช้ 2 วิธีเทียบกัน
- **IQR** (ต่ำกว่า Q1 − 1.5×IQR หรือสูงกว่า Q3 + 1.5×IQR) เหมาะกับข้อมูลเบ้
- **Z-score** (|z| > 3) เหมาะกับข้อมูลรูประฆัง

In [ ]:
KEY = ["per_capita_electricity", "renewables_share_elec", "gas_share_elec", "coal_share_elec",
       "carbon_intensity_elec", "electricity_generation", "net_elec_imports"]

def iqr_mask(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return (s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)

rows = []
for c in KEY:
    s = y[c].dropna()
    m_iqr, m_z = iqr_mask(s), ((s - s.mean()) / s.std()).abs() > 3
    top = y.loc[s[m_iqr].sort_values(ascending=False).index, "country"].head(4).tolist()
    rows.append({"ตัวแปร": c, "n": len(s), "skew": round(s.skew(), 1),
                 "IQR พบ": int(m_iqr.sum()), "Z-score พบ": int(m_z.sum()),
                 "% ผิดปกติ (IQR)": round(m_iqr.mean() * 100, 1), "ตัวอย่าง": ", ".join(top)})
out_tab = pd.DataFrame(rows)
out_tab.to_csv(os.path.join(OUT, "outliers_2021.csv"), index=False, encoding="utf-8-sig")
out_tab

**อ่านตาราง:**
- **ตัวแปรที่ได้รับผลกระทบมาก:** `electricity_generation` (ประเทศใหญ่), `per_capita_electricity` (ประเทศรวย/อุตสาหกรรมหนัก), `net_elec_imports` (ผู้ส่งออก/นำเข้ารายใหญ่)
- **ไม่มีค่าผิดปกติเลย:** สัดส่วนหมุนเวียน และความเข้มคาร์บอน เพราะกระจายกว้างทั่วช่วง จึงไม่มีค่าไหนหลุดกลุ่ม
- **ระวัง! IQR "หลอก" กับข้อมูลที่ส่วนใหญ่เป็น 0:** สัดส่วนถ่านหิน (17%), สัดส่วนก๊าซ (6%) และไฟฟ้านำเข้าสุทธิ (33%) ถูกตีว่าผิดปกติจำนวนมาก เพราะประเทศส่วนใหญ่มีค่าใกล้ 0 ทำให้ IQR แคบมาก ประเทศที่ใช้ถ่านหินหรือก๊าซจริงจังจึงดู "ผิดปกติ" ทั้งที่เป็นเรื่องปกติ (ไทยก็ถูกตีว่าผิดปกติในตัวแปรนำเข้าไฟฟ้า) → สำหรับข้อมูลแบบนี้ ต้องใช้ **เหตุผลเชิงธุรกิจ** ตัดสิน ไม่ใช่สูตรอย่างเดียว
- **Z-score พบน้อยกว่า IQR** ทุกตัวแปรที่เบ้ เพราะค่าสุดโต่งทำให้ std ใหญ่ขึ้นจนตัวเองไม่ดูผิดปกติ

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, c, lg in zip(ax, ["per_capita_electricity", "electricity_generation", "net_elec_imports"], [True, True, False]):
    s = y[c].dropna()
    if lg:
        s = np.log10(s[s > 0])
    a.boxplot(s, vert=False, widths=0.5)
    m = iqr_mask(s)
    for v, name in zip(s[m], y.loc[s[m].index, "country"]):
        if abs(v - s.median()) >= np.sort(np.abs(s[m] - s.median()))[-3]:   # ติดชื่อ 3 จุดที่ไกลสุด
            a.annotate(name, (v, 1), xytext=(0, 8), textcoords="offset points", fontsize=8, ha="center", rotation=30)
    a.set_title(c + (" (log10)" if lg else " (TWh)")); a.set_yticks([])
plt.tight_layout(); savefig("outlier_boxplots.png"); plt.show()

## 3.2 ค่าผิดปกติกระทบการวิเคราะห์แค่ไหน
ทดลองคำนวณสถิติ 2 แบบ: **รวม** กับ **ตัดค่าผิดปกติออก** เพื่อดูว่าสถิติตัวไหน "ไว" ต่อค่าผิดปกติ

In [ ]:
imp = []
for c in ["per_capita_electricity", "electricity_generation", "coal_share_elec"]:
    s = y[c].dropna(); k = s[~iqr_mask(s)]
    imp.append({"ตัวแปร": c, "mean รวม": s.mean(), "mean ตัดออก": k.mean(),
                "Δ mean (%)": (k.mean() / s.mean() - 1) * 100,
                "median รวม": s.median(), "median ตัดออก": k.median(),
                "Δ median (%)": (k.median() / s.median() - 1) * 100 if s.median() else np.nan})
pd.DataFrame(imp).round(1)

**อ่านผล:** การตัดค่าผิดปกติทำให้ **mean เปลี่ยนมาก** (ไฟฟ้าต่อคน −21%, ไฟฟ้ารวม −87%) แต่ **median เปลี่ยนน้อยกว่ามาก** (ไฟฟ้าต่อคน −5%) ส่วนสัดส่วนถ่านหิน ถ้าตัด "ค่าผิดปกติหลอก" ออก ค่าเฉลี่ยจะเหลือเกือบ 0 ซึ่งผิดความจริง
→ วิธีรับมือที่เลือก: **เก็บค่าจริงไว้** (เป็นข้อมูลถูกต้อง) แต่ **รายงานด้วย median** และ **แสดงกราฟด้วยสเกล log** ซึ่งแดชบอร์ดทำตามนี้แล้ว

## 3.3 รูปแบบการขาดหาย (Missing patterns)
### % ว่างต่อคอลัมน์

In [ ]:
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(miss.index[::-1], miss.values[::-1], color=["#B3412E" if v > 5 else "#9AA5A8" for v in miss.values[::-1]])
ax.axvline(5, color="black", ls="--", lw=1); ax.set_xlabel("% missing (dashed = 5%)")
ax.set_title("Missing % per column (clean data)"); plt.tight_layout(); savefig("missing_pct.png"); plt.show()
print("คอลัมน์ที่ว่างเกิน 5%:", miss[miss > 5].round(1).to_dict())

### ขาดเมื่อไร: % ว่างแยกตามปี
ถ้าค่าว่างกระจุกบางปี แปลว่าเป็นปัญหาการเก็บข้อมูลของแหล่ง ไม่ใช่ปัญหาของประเทศ

In [ ]:
cols = ["gdp", "electricity_generation", "carbon_intensity_elec", "electricity_demand", "renewables_share_elec", "population"]
by_year = df.groupby("year")[cols].apply(lambda g: g.isna().mean() * 100)
fig, ax = plt.subplots(figsize=(13, 3.6))
im = ax.imshow(by_year.T, aspect="auto", cmap="Reds", vmin=0, vmax=100)
ax.set_yticks(range(len(cols)), cols); ax.set_xticks(range(len(by_year)), by_year.index, rotation=90)
plt.colorbar(im, label="% missing"); ax.set_title("Missing % by year (darker = more missing)")
plt.tight_layout(); savefig("missing_by_year.png"); plt.show()

**อ่านผล:** แถบสีเข้มมี 2 จุดเท่านั้น
1. **GDP หลังปี 2018** ว่างทั้งหมด เพราะแหล่งข้อมูลหยุดเก็บ (ขาดตามเวลา ไม่เกี่ยวกับประเทศ)
2. **ปี 2022** ว่างหลายคอลัมน์ เพราะเป็นข้อมูลเบื้องต้น
ปีอื่นแทบไม่มีค่าว่าง

### ขาดร่วมกันไหม (Co-missingness)
ถ้าคอลัมน์ A ว่างแล้ว B มักว่างด้วย = **ขาดร่วมกัน** มักเกิดจากสาเหตุเดียวกัน
วัดด้วยสหสัมพันธ์ของ "ตัวบ่งชี้ว่าง" (1 = ว่าง, 0 = มีค่า) ค่าใกล้ 1 = ว่างพร้อมกันเสมอ

In [ ]:
mcols = [c for c in df.columns if df[c].isna().any()]
ind = df[mcols].isna().astype(int)
co = ind.corr()
short = {c: c.replace("_electricity", "_elec").replace("_share_elec", "_sh").replace("other_renewables_sh_exc_biofuel", "other_ren_sh")
         .replace("other_renewable_exc_biofuel_elec", "other_ren_elec") for c in mcols}
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(co, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(len(mcols)), [short[c] for c in mcols], rotation=90, fontsize=7)
ax.set_yticks(range(len(mcols)), [short[c] for c in mcols], fontsize=7)
plt.colorbar(im, fraction=0.046, label="correlation of missingness")
ax.set_title("Co-missingness: which columns are missing together"); plt.tight_layout()
savefig("co_missing.png"); plt.show()

# รูปแบบการขาดที่พบบ่อย (แต่ละแถวขาดคอลัมน์ไหนบ้าง)
pattern = ind.apply(lambda r: ", ".join(short[c] for c in mcols if r[c]) or "(ครบทุกคอลัมน์)", axis=1)
pattern.value_counts().head(6).to_frame("จำนวนแถว")

In [ ]:
# ค่าว่างที่เหลือ (ไม่นับ GDP และปี 2022) อยู่ที่ประเทศไหน
rest = df[(df["year"] <= LATEST) & df["electricity_generation"].isna()]
print("แถวที่ไม่มีข้อมูลไฟฟ้า (ปี 2000-2021):", len(rest))
print(rest.groupby("country")["year"].agg(["count", "min", "max"]).sort_values("count", ascending=False).head(8))

**อ่านผล:**
- คอลัมน์ไฟฟ้าทุกตัว **ว่างพร้อมกัน** (สีม่วงเข้มเป็นบล็อกใหญ่) แปลว่าถ้าประเทศไม่รายงานไฟฟ้าปีนั้น ก็ไม่มีข้อมูลไฟฟ้าทุกตัว ไม่ใช่ว่างกระจัดกระจาย
- รูปแบบที่พบบ่อยที่สุดคือ **ว่างแค่ GDP** (ปี 2019–2022) รองลงมาคือ **ว่างทั้งชุดไฟฟ้า**
- แถวที่ไม่มีข้อมูลไฟฟ้าในปี 2000–2021 มีแค่ 53 แถว และเกือบทั้งหมดเป็น **เขตปกครองโพ้นทะเล** (Western Sahara และดินแดนของฝรั่งเศส เช่น French Guiana, Reunion) ซึ่งเป็นแถวที่เราเปลี่ยนจาก 0 ปลอมเป็นค่าว่างในขั้นทำความสะอาดที่ 2
- ค่าว่างเกิดจาก **ช่วงเวลา** และ **ขนาดประเทศ** เป็นระบบ ไม่ใช่สุ่ม → ถ้าเติมค่าแบบสุ่มหรือเติมด้วยค่าเฉลี่ยโลกจะผิด

## 3.4 วิธีรับมือ และผลกระทบ
| ปัญหา | วิธีรับมือ | ผลกระทบต่อการวิเคราะห์ |
|---|---|---|
| ค่าสูงจริง (ไฟฟ้าต่อคน, ไฟฟ้ารวม) | เก็บไว้ + ใช้ median + สเกล log | ค่ากลางไม่เพี้ยน กราฟอ่านได้ |
| GDP ว่างหลัง 2018 | ไม่เติม วิเคราะห์ GDP ถึง 2018 เท่านั้น | คำถามเรื่อง GDP ตอบได้ถึงปี 2018 |
| ปี 2022 ข้อมูลไม่ครบ | แยกกลุ่ม: ภาพรวมโลกใช้ถึงปี 2021 | ไม่มีปีล่าสุดสำหรับภาพรวมโลก |
| เขตปกครองเล็กไม่มีข้อมูลบางปี | ปล่อยว่าง ไม่เติม | ไม่กระทบคำถามเรื่องไทย/อาเซียน |

## ✅ สรุปขั้นนี้ (เกณฑ์ 3.5.2.3)
| ข้อย่อย | หลักฐาน |
|---|---|
| 3.5.2.3.1 ตรวจ outliers เกณฑ์ชัด + ชี้ตัวแปรที่กระทบ | ข้อ 3.1 `outliers_2021.csv`, `outlier_boxplots.png` |
| 3.5.2.3.2 %missing + ภาพการขาดร่วมกัน | ข้อ 3.3 `missing_pct.png`, `missing_by_year.png`, `co_missing.png` |
| 3.5.2.3.3 วิธีรับมือและผลกระทบ | ข้อ 3.2 และ 3.4 |

**ไฟล์ถัดไป:** `eda_step4_insights.ipynb` ตอบคำถามจาก `step1_questions` และสรุป Insight